# Calibration and repeatability — Plots 9 and 4

Why should anyone trust the ellipse on the map? This notebook answers that in three steps.

1. **Jitter:** 20 knocks at one taped spot. The spread of the time gap ÷ √2 is the real per-sensor timing error, which goes into `timing.sigma_s` in the array file.
2. **Calibration (Plot 9):** knocks at taped spots with known positions. Each sensor has a small fixed delay; we fit it, subtract it, and check the timing residuals shrink. This writes `config/channel_delays.yaml`, which `pipeline.py` uses automatically.
3. **Repeatability (Plot 4):** 10+ knocks at a spot **not used for calibration**. If about 95% of them land inside their own 95% ellipse/wedge, the uncertainty is honest.

**Recording protocol (one recording per spot, every knock from that spot):**
```
python pipeline.py data/recordings/<file>.npz --run cal-T1 --array <array> --true-spot T1 --delays none
```
`--delays none` keeps calibration runs uncalibrated. Spots are in `config/cal_spots.yaml`.

With `SIMULATE = True`, the notebook synthesizes those exact sessions with known delays and jitter, so the method can be checked before any knocking.

In [ ]:
# ===================== EDIT THESE =====================
MODE = "direction"                 # "direction" (2-sensor pair) or "triangle" (3 sensors)
SIMULATE = True                    # True: synthesize the sessions below; False: use your recorded runs

if MODE == "direction":
    ARRAY = "config/array.yaml"
    CAL_RUNS = {"cal-dir-P90": ("P90", 20)}          # run name: (taped spot, knocks)
    TEST_RUN = ("rep-dir-P45", "P45", 20)            # held-out spot: jitter + repeatability
else:
    ARRAY = "config/array_triangle.yaml"
    CAL_RUNS = {"cal-tri-T1": ("T1", 5), "cal-tri-T3": ("T3", 5), "cal-tri-T4": ("T4", 5)}
    TEST_RUN = ("rep-tri-T2", "T2", 10)

SIM_JITTER_S = 100e-6              # sim only: true per-sensor jitter (try 200e-6 to see "overconfident")
SIM_DELAYS_S = (0.0, 120e-6, -80e-6)
# ======================================================

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, yaml, matplotlib.pyplot as plt
import calibrate as C
from arraycfg import load_array

cfg = load_array(ARRAY); names = cfg["_names"]; n_ch = len(names)
if SIMULATE:
    for i, (run, (spot, n)) in enumerate(CAL_RUNS.items()):
        C.make_spot_session(ARRAY, run, spot, n, seed=10 + i, jitter_s=SIM_JITTER_S, delays_s=SIM_DELAYS_S)
    C.make_spot_session(ARRAY, TEST_RUN[0], TEST_RUN[1], TEST_RUN[2], seed=99, jitter_s=SIM_JITTER_S, delays_s=SIM_DELAYS_S)
    print("simulated", list(CAL_RUNS), "and", TEST_RUN[0])
print(f"{MODE}: {n_ch} sensors, assumed timing.sigma_s = {cfg['timing']['sigma_s']*1e6:.0f} us")

## 1. Jitter

In [ ]:
jit = C.measure_jitter(C.gap_table([TEST_RUN[0]], ARRAY), n_ch)
print(f"{jit['n']} knocks at {TEST_RUN[1]}: per-sensor jitter = {jit['sigma_s']*1e6:.0f} us "
      f"(array file assumes {cfg['timing']['sigma_s']*1e6:.0f} us)")
print(f"-> put  timing: {{sigma_s: {jit['sigma_s']:.6f}}}  in {ARRAY}" if not SIMULATE else
      f"   (sim truth: {SIM_JITTER_S*1e6:.0f} us of sensor jitter + the detector's own few us)")

## 2. Calibration: per-sensor delays (Plot 9)

In [ ]:
gaps = C.gap_table(list(CAL_RUNS), ARRAY)
fit = C.fit_delays(gaps, n_ch)
for k in range(n_ch):
    print(f"{names[k]}: {fit['delays_s'][k]*1e6:+7.1f} us  ± {2*fit['se_s'][k]*1e6:.1f} (2 SE)" + ("  reference" if k == 0 else ""))
title = (f"Fitted delays cut timing residuals from {fit['rms_before_us']:.0f} to {fit['rms_after_us']:.0f} µs RMS "
         f"({fit['n']} knocks at {', '.join(s for s, _ in CAL_RUNS.values())})")
fig = C.plot_calibration(fit, names, SIM_DELAYS_S[:n_ch] if SIMULATE else None, title)
fig.savefig(ROOT / f"docs/figures/plot9_calibration_{MODE}.png", dpi=130); plt.show()
out = C.write_delays(fit, ARRAY, list(CAL_RUNS), "config/channel_delays_sim.yaml" if SIMULATE else "config/channel_delays.yaml")
print("wrote", out.relative_to(ROOT), "(sim fits never overwrite the real file)" if SIMULATE else "")
print("Rule of thumb: delay uncertainty ≈ gap spread / sqrt(knocks). With ~140 us spread, 20 knocks pin a delay to ±30 us.")

## 3. Repeatability: is the uncertainty honest? (Plot 4)

In [ ]:
fixes, raw, st = C.repeatability(TEST_RUN[0], ARRAY, str(out))
what = "ellipse" if fixes.kind.iloc[0] == "position" else "±2σ wedge"
title = (f"{st['inside']} of {st['n']} knocks at {TEST_RUN[1]} inside their own 95% {what}: {st['verdict']}")
fig = C.plot_repeatability(fixes, raw, st, ARRAY, title)
fig.savefig(ROOT / f"docs/figures/plot4_repeatability_{MODE}.png", dpi=130); plt.show()
print("VERDICT:", st["verdict"])
print(f"evidence: {st['inside']}/{st['n']} inside (an honest 95% region keeps at least {st['k_min']} with 99% probability); "
      f"error scale {st['scale']:.2f} (1.0 = honest, >1.3 overconfident, <0.5 conservative)")

### If the verdict is "overconfident"

In order of likelihood:
1. **`timing.sigma_s` is too small.** Set it to the jitter measured in step 1 and rerun.
2. **Delays are stale or poorly fitted.** Refit with more knocks (20+) or more spots; refit after moving or rewiring any sensor.
3. **Sensor positions are off.** A 2 cm tape error looks exactly like a timing bias. Re-measure.
4. **Clock not shared** between sensors (separate boards): no calibration fixes drift; that needs time sync.

If it says "conservative", the regions are bigger than they need to be: lower `timing.sigma_s` toward the measured jitter.